# Unit 5 — 06: Model Versioning and Reproducibility

**What you will do:** Demonstrate why reproducibility fails without discipline, fix it with seeds and metadata, version a model with a full audit trail, hash a dataset, pin environment versions, and log the git commit hash alongside a model.

**Why it matters:** If you cannot reproduce a model, you cannot debug production incidents, audit decisions, or compare experiments fairly.

**How to run:** Python 3.10+. Run cells in order.

## 🔗 Where this fits

**Builds on:** Course 05 (AIAT 115) — Unit 1, lesson 05 "05. Jupyter Notebooks Best Practices" — reproducibility as notebook hygiene (run order, a clean restart); here it becomes auditable: a fixed seed, a hash of the dataset, pinned library versions and the git commit stored beside the model.

---

## 📰 A Nature paper that could not be reproduced

On **1 January 2020** a Google Health team published a breast-cancer screening study in *Nature* (McKinney et al., *Nature* 577, 89–94) reporting an AI system that outperformed radiologists. In **October 2020** the same journal carried a response from **Benjamin Haibe-Kains and 30 co-authors**, *Transparency and reproducibility in artificial intelligence* (*Nature* 586, E14–E16, published online 14 October 2020), arguing that the study could not be independently verified: the model code and much of the methodological detail were not released, and key datasets were not accessible. Their objection was not about the result. It was about what a result *is* if no one else can regenerate it.

Deployment raises exactly the same question with money and liability attached. When a production model produces a decision somebody disputes six months from now, "we trained a random forest on our customer data last spring" is not an answer. The artifact, the data, the environment and the code have to be pinned together tightly enough that you can rebuild the exact model that made that exact decision.

**What goes wrong without this lesson.** You cannot debug what you cannot reproduce. Without a seed you get a different model each run; without a data hash you cannot tell whether the data changed; without pinned versions you cannot rebuild last year's environment; without a commit hash you cannot find the code. Each gap alone turns a two-hour investigation into a week of guessing.

## Section 1 — The Reproducibility Problem

"Same code + same data" does **not** guarantee the same model if any of these differ:

1. **Random seeds** — sklearn, numpy, and Python's `random` each have their own RNG
2. **Library versions** — sklearn 1.3 and 1.4 may produce slightly different trees
3. **Hardware** — floating-point operations on GPU are not fully deterministic by default
4. **Data order** — if rows are loaded in a different order, batches differ

Full reproducibility requires locking all four.

## Section 2 — Seeds and Determinism

Show concretely that training without a seed produces different models each run, while with a seed the results are identical.

In [1]:
# WHAT: demonstrate seeding — train pairs of forests without and with a fixed
# random_state on a deliberately noisy dataset.
# WHY: reproducibility starts here: unseeded runs disagree with themselves,
# which makes every later comparison (and every bug report) unverifiable.
import random
import numpy as np
import pathlib
import hashlib
import json
import subprocess
import sys
from dataclasses import dataclass, asdict
from datetime import datetime
from typing import List, Optional

import joblib
from sklearn.datasets import load_iris, make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

SEED = 42

# Iris is used by the versioning / hashing sections later in this notebook.
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# For the SEED demo we use a deliberately NOISY dataset. Iris is so easy that two
# unseeded forests still reach identical predictions, which would hide the effect
# we are trying to show. With label noise and overlapping classes, the randomness
# in feature/bootstrap sampling actually changes the predictions.
X_demo, y_demo = make_classification(
    n_samples=400, n_features=10, n_informative=4, n_redundant=2,
    n_classes=3, flip_y=0.15, class_sep=0.7, random_state=0,
)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_demo, y_demo, test_size=0.3, random_state=42
)

# --- WITHOUT a seed: results differ between runs ---
model_a = RandomForestClassifier(n_estimators=10)  # no random_state
model_b = RandomForestClassifier(n_estimators=10)  # no random_state
model_a.fit(Xd_train, yd_train)
model_b.fit(Xd_train, yd_train)
pred_a = model_a.predict(Xd_test)
pred_b = model_b.predict(Xd_test)
print("WITHOUT seed:")
print(f"  Model A accuracy: {(pred_a == yd_test).mean():.4f}")
print(f"  Model B accuracy: {(pred_b == yd_test).mean():.4f}")
print(f"  Predictions identical: {np.array_equal(pred_a, pred_b)}  <- expect False")

print()

# --- WITH a seed: results match exactly ---
random.seed(SEED)
np.random.seed(SEED)

model_c = RandomForestClassifier(n_estimators=10, random_state=SEED)
model_d = RandomForestClassifier(n_estimators=10, random_state=SEED)
model_c.fit(Xd_train, yd_train)
model_d.fit(Xd_train, yd_train)
pred_c = model_c.predict(Xd_test)
pred_d = model_d.predict(Xd_test)
print("WITH seed=42:")
print(f"  Model C accuracy: {(pred_c == yd_test).mean():.4f}")
print(f"  Model D accuracy: {(pred_d == yd_test).mean():.4f}")
print(f"  Predictions identical: {np.array_equal(pred_c, pred_d)}  <- expect True")

WITHOUT seed:
  Model A accuracy: 0.6167
  Model B accuracy: 0.5750
  Predictions identical: False  <- expect False

WITH seed=42:
  Model C accuracy: 0.6083
  Model D accuracy: 0.6083
  Predictions identical: True  <- expect True


## Section 3 — ModelVersion Dataclass and Versioned Save/Load

A good versioning system saves the model **and** a JSON sidecar with everything needed to reproduce and audit it.

In [2]:
# WHAT: save a model together with a metadata sidecar (version, dates, library
# versions, accuracy, seed) via a ModelVersion dataclass.
# WHY: a bare .pkl answers nothing in six months — the JSON sidecar is the
# difference between an artifact and a mystery file.
import sklearn


@dataclass
# The metadata schema: everything needed to reproduce or audit this model.
class ModelVersion:
    version_id: str
    model_path: str
    training_date: str
    sklearn_version: str
    python_version: str
    accuracy: float
    feature_names: List[str]
    random_seed: int


MODEL_DIR = pathlib.Path("/tmp/versioned_models")
MODEL_DIR.mkdir(exist_ok=True)


# Saves TWO files per version: the .pkl artifact and its .json sidecar.
def save_versioned_model(
    model: object,
    X_test: np.ndarray,
    y_test: np.ndarray,
    version_id: str,
    feature_names: List[str],
    random_seed: int,
) -> ModelVersion:
    """Save model + metadata JSON sidecar. Returns the ModelVersion record."""
    model_path = MODEL_DIR / f"{version_id}.pkl"
    meta_path  = MODEL_DIR / f"{version_id}.json"

    # Artifact first, then measure accuracy on the spot so metadata is honest.
    joblib.dump(model, model_path)

    accuracy = float(model.score(X_test, y_test))
    version = ModelVersion(
        version_id=version_id,
        model_path=str(model_path),
        training_date=datetime.utcnow().isoformat(),
        sklearn_version=sklearn.__version__,
        python_version=f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
        accuracy=round(accuracy, 4),
        feature_names=feature_names,
        random_seed=random_seed,
    )
    with open(meta_path, "w") as f:
        json.dump(asdict(version), f, indent=2)

    print(f"Saved model:    {model_path}")
    print(f"Saved metadata: {meta_path}")
    return version


# The loader returns BOTH the model and its metadata — they travel as a pair.
def load_versioned_model(version_id: str) -> tuple:
    """Load model and metadata. Returns (model, ModelVersion)."""
    model_path = MODEL_DIR / f"{version_id}.pkl"
    meta_path  = MODEL_DIR / f"{version_id}.json"
    model = joblib.load(model_path)
    with open(meta_path) as f:
        meta = ModelVersion(**json.load(f))
    return model, meta


# Save the seeded model
production_model = RandomForestClassifier(n_estimators=50, random_state=SEED)
production_model.fit(X_train, y_train)

version = save_versioned_model(
    model=production_model,
    X_test=X_test,
    y_test=y_test,
    version_id="v1.0.0",
    feature_names=list(iris.feature_names),
    random_seed=SEED,
)
print(f"\nVersion record: {json.dumps(asdict(version), indent=2)}")

Saved model:    /tmp/versioned_models/v1.0.0.pkl
Saved metadata: /tmp/versioned_models/v1.0.0.json

Version record: {
  "version_id": "v1.0.0",
  "model_path": "/tmp/versioned_models/v1.0.0.pkl",
  "training_date": "2026-08-23T16:53:50.190958",
  "sklearn_version": "1.8.0",
  "python_version": "3.14.3",
  "accuracy": 1.0,
  "feature_names": [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
  ],
  "random_seed": 42
}


/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_17023/1738974425.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  training_date=datetime.utcnow().isoformat(),


In [3]:
# Load and verify
loaded_model, loaded_meta = load_versioned_model("v1.0.0")
print(f"Loaded model: {type(loaded_model).__name__}")
print(f"Metadata:")
for k, v in asdict(loaded_meta).items():
    print(f"  {k}: {v}")

Loaded model: RandomForestClassifier
Metadata:
  version_id: v1.0.0
  model_path: /tmp/versioned_models/v1.0.0.pkl
  training_date: 2026-08-23T16:53:50.190958
  sklearn_version: 1.8.0
  python_version: 3.14.3
  accuracy: 1.0
  feature_names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
  random_seed: 42


## Section 4 — Data Versioning with Hashing

If you change even one row in your training data, the hash changes. This is how you detect silent data mutations.

In [4]:
# WHAT: hash the training data and show the hash changing when one cell mutates.
# WHY: 'same code, same model' still breaks if the DATA silently changed —
# a stored data hash makes that silent change loud.
def compute_data_hash(df: pd.DataFrame) -> str:
    """Return an MD5 hash of the dataframe contents. Changes if any value changes."""
    row_hashes = pd.util.hash_pandas_object(df, index=True).values
    return hashlib.md5(row_hashes.tobytes()).hexdigest()


# Create a DataFrame from the iris dataset
df_iris = pd.DataFrame(X, columns=iris.feature_names)
df_iris["target"] = y

hash_v1 = compute_data_hash(df_iris)
print(f"Original dataset hash:   {hash_v1}")

# Simulate a data mutation (one value changed)
df_mutated = df_iris.copy()
df_mutated.iloc[0, 0] = 999.0  # corrupt one cell
hash_mutated = compute_data_hash(df_mutated)
print(f"After mutation hash:     {hash_mutated}")
print(f"Hash changed: {hash_v1 != hash_mutated}")

print("\nIn production: store the data hash alongside the model version.")
print("If the hash changes between training runs, you know the data changed.")

Original dataset hash:   fa23f54d92bca516aa06bb08c256935a
After mutation hash:     98128e15eeb0ab14ce4271a1d42d813f
Hash changed: True

In production: store the data hash alongside the model version.
If the hash changes between training runs, you know the data changed.


## 💬 Discuss

The seeding experiment is unusually clear. Without a seed, two identical training calls gave **0.6167** and **0.5750** — a 4-point accuracy gap from randomness alone, and `Predictions identical: False`. With `random_state=42`, both runs gave **0.6083** and predictions matched exactly.

1. A colleague reports their new model scores 0.6167 against your 0.5750 and proposes shipping theirs. Using only the numbers above, explain what is wrong with that comparison — and describe the experiment you would run instead to decide whether the difference is real.
2. Seeding makes runs identical *and* fixes the model to one draw from a distribution. What does that hide, and what would you report alongside a seeded result so a reader knows how stable it is?
3. The audit record combines git commit, data hash, library versions, seed and accuracy. Is that enough to rebuild the model bit-for-bit on a different machine? Work through what is still missing — and decide how much of the gap is worth closing for a model that declines loan applications.

## Section 5 — Environment Reproducibility

Pin exact library versions so that anyone can recreate the environment that produced a specific model.

In [5]:
# WHAT: read the installed versions of the key libraries and write a pinned
# requirements file next to the model version.
# WHY: sklearn 1.3 and 1.4 can produce different models from identical code —
# pinning freezes the third ingredient of reproducibility: the environment.
from importlib.metadata import version as _pkg_version, PackageNotFoundError

PACKAGES_TO_PIN = ["scikit-learn", "numpy", "pandas", "joblib", "scipy"]

pinned = []
print("Pinned environment (save as requirements.txt):")
print("-" * 40)
for pkg_name in PACKAGES_TO_PIN:
    try:
        version = _pkg_version(pkg_name)
        line = f"{pkg_name}=={version}"
        pinned.append(line)
        print(line)
    except PackageNotFoundError:
        print(f"{pkg_name}: NOT INSTALLED")

# Save to file
requirements_path = pathlib.Path("/tmp/versioned_models/requirements_v1.0.0.txt")
requirements_path.write_text("\n".join(pinned) + "\n")
print(f"\nSaved to: {requirements_path}")
print("\nTo recreate: pip install -r requirements_v1.0.0.txt")

Pinned environment (save as requirements.txt):
----------------------------------------
scikit-learn==1.8.0
numpy==2.4.4
pandas==2.3.3
joblib==1.5.3
scipy==1.17.1

Saved to: /tmp/versioned_models/requirements_v1.0.0.txt

To recreate: pip install -r requirements_v1.0.0.txt


## Section 6 — Git Integration

Log the exact git commit hash alongside a model so you always know which code produced it.

In [6]:
# WHAT: capture the git commit and combine EVERYTHING — code, data hash,
# environment, seed, accuracy — into one audit record.
# WHY: this single JSON answers the auditor's question 'exactly how was this
# model built?' — the end goal of the whole lesson.
def get_git_commit_hash() -> Optional[str]:
    """Return the current HEAD commit hash, or None if not in a git repo."""
    try:
        result = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL,
        )
        return result.decode().strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


git_hash = get_git_commit_hash()
if git_hash:
    print(f"Git commit: {git_hash}")
else:
    git_hash = "not-a-git-repo"
    print("Not in a git repo — in production this would always have a commit hash.")

# Combine all provenance information into one audit record
audit_record = {
    "model_version": "v1.0.0",
    "git_commit": git_hash,
    "data_hash": hash_v1,
    "sklearn_version": sklearn.__version__,
    "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
    "random_seed": SEED,
    "training_date": datetime.utcnow().isoformat(),
    "accuracy_on_test": round(production_model.score(X_test, y_test), 4),
}

audit_path = pathlib.Path("/tmp/versioned_models/audit_v1.0.0.json")
audit_path.write_text(json.dumps(audit_record, indent=2))

print("\nComplete audit record:")
print(json.dumps(audit_record, indent=2))
print(f"\nSaved to: {audit_path}")

Git commit: d53fd99edf8518dab5fda9ad74e60c50d3bcee8e

Complete audit record:
{
  "model_version": "v1.0.0",
  "git_commit": "d53fd99edf8518dab5fda9ad74e60c50d3bcee8e",
  "data_hash": "fa23f54d92bca516aa06bb08c256935a",
  "sklearn_version": "1.8.0",
  "python_version": "3.14.3",
  "random_seed": 42,
  "training_date": "2026-08-23T16:53:50.225844",
  "accuracy_on_test": 1.0
}

Saved to: /tmp/versioned_models/audit_v1.0.0.json


/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_17023/3577273463.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "training_date": datetime.utcnow().isoformat(),


## Summary

Three things to version together for every model release:

1. **Model artifact** — the `.pkl` file + metadata JSON (accuracy, feature names, seed, sklearn/python versions)
2. **Data** — the MD5 hash of the training dataset, so you can detect if data changed between runs
3. **Environment** — a pinned `requirements.txt` so the exact library versions can be restored

Plus: log the **git commit hash** to tie the model to the exact code that trained it.

## Self-Check (answer before scrolling back up)

1. **Why isn't setting `random_state` in sklearn enough to guarantee full reproducibility?**  
   `random_state` only controls sklearn's internal RNG. NumPy operations outside sklearn, Python's `random` module, and thread-level nondeterminism are not controlled. On GPU, floating-point operations may be reordered. You also need to pin library versions: a different version of sklearn may implement the same algorithm differently.

2. **What is a data hash and when would it change?**  
   A data hash (e.g., MD5 of row hashes) is a fixed-length fingerprint of the entire dataset. It changes whenever any row is added, removed, or any value is modified — even a single digit in one cell. It acts as a tamper-evident seal on your training data.

3. **If you can reproduce a model exactly, what does that enable you to do with incidents?**  
   You can bisect: reproduce the exact model that is in production, run it on the failing input, and confirm you see the same bad output. Then modify each component (data, code, seed) one at a time to find the root cause. Without reproducibility, you cannot isolate whether the bug is in the code, the data, or an environment difference.

## ⚠️ Where this breaks

- **A seed fixes the draw, not the outcome across environments.** `random_state=42` makes two runs on *this* machine identical. It does not survive a different scikit-learn version, a different BLAS/threading configuration, or a different CPU — floating-point operations may be reordered and results may differ in the last digits, which is sometimes enough to flip a borderline prediction. Reproducibility across machines is a stronger claim than reproducibility across runs, and this notebook only demonstrates the second.
- **Hashing the data proves it changed; it does not tell you how.** An MD5 that differs tells you something moved. It does not say which rows, whether the change was legitimate, or whether the new version is better. Pair the hash with a row count and per-column summary statistics if you want the hash to be actionable.
- **Pinned versions rot.** `requirements_v1.0.0.txt` records what you had. In two years some of those versions may be yanked from PyPI, or refuse to build on a current OS. If you must genuinely rebuild years later, the artifact that survives is a **container image**, not a requirements file (Unit 4, notebook 01).
- **The assumption that must hold:** the git commit describes the code that ran. Uncommitted local edits, a dirty working tree, or a notebook run out of order all make the recorded hash a plausible lie. Record whether the tree was dirty, and prefer training from committed code (Unit 4, notebook 04).
- **Reproducible is not explainable.** You can rebuild a model exactly and still be unable to tell an applicant why they were declined. The Google Health dispute above was about reproducibility; a customer complaint is usually about explanation. They are different obligations requiring different tools (Course 06).
- **The cheaper alternative.** For an exploratory model that will never leave a notebook, a seed and a `requirements.txt` is proportionate. The full audit record — hash, commit, pinned environment, sidecar metadata — is the price of a model whose decisions someone can dispute. Pay it for those; do not pay it for everything.

## 📚 References

1. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
2. Pineau, J., Vincent-Lamarre, P., Sinha, K., et al. (2021). *Improving Reproducibility in Machine Learning Research (A Report from the NeurIPS 2019 Reproducibility Program)*. JMLR 22. <https://arxiv.org/abs/2003.12206>
3. Boettiger, C. (2015). *An Introduction to Docker for Reproducible Research*. ACM SIGOPS Operating Systems Review 49(1). <https://arxiv.org/abs/1410.0846>
